# PyMOL -> Blender runner (with Blender-side remote control)
This notebook runs export/build, then launches Blender with a `bpy` startup script that unhides/focuses `1A3N` and switches to camera view.

In [ ]:
from pathlib import Path
import subprocess

project_dir = Path('/Users/mb/Documents/blender_bridge')
blender = Path('/Applications/Blender.app/Contents/MacOS/Blender')
scene = project_dir / '1A3N_scene.blend'
remote_script = project_dir / 'scripts' / 'remote_open_focus.py'

print('project_dir  =', project_dir)
print('blender      =', blender)
print('scene        =', scene)
print('remote_script=', remote_script)


In [ ]:
commands = [
    ['pixi', 'run', 'export-1a3n'],
    ['pixi', 'run', 'build-1a3n'],
]

for cmd in commands:
    print('\n$ ' + ' '.join(cmd))
    result = subprocess.run(cmd, cwd=project_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr.strip())
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')

print('\nBuild complete.')


In [ ]:
if not blender.exists():
    raise FileNotFoundError(f'Blender executable not found: {blender}')
if not scene.exists():
    raise FileNotFoundError(f'Scene file not found: {scene}')
if not remote_script.exists():
    raise FileNotFoundError(f'Remote control script not found: {remote_script}')

cmd = [
    'open', '-a', str(blender.parent.parent), '--args',
    '--python', str(remote_script),
    '--', str(scene), '1A3N'
]
print('$ ' + ' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
print(f'Started Blender launcher (pid={proc.pid})')
print('The script runs inside Blender (bpy), applies focus/visibility/camera, then keeps Blender open.')
